# Darcy face-jump indicator

Equations (5.1)–(5.3) of Araya et al. (2013), [DOI 10.1137/120888223](https://doi.org/10.1137/120888223), measure broken pressure jumps. Exact local lifts are assumptions of the reliability theorem; this notebook uses finite P2 local spaces and does not infer a certified bound from the jump indicator alone.


In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


In [ ]:
import json
from pathlib import Path
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm import TriangleMesh
from pymhm.estimators.darcy_jump import estimate_darcy_jumps


## Declare the local and global equations

The application writes local A, B and C blocks with the shared FEM kernels,
then supplies them to the generic assembler. Primal coefficients satisfy

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T,\\
L_T(v)&=(f,v)_T+\sum_j\alpha_jv(x_j),\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tp&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

Signs follow the fixed macroface normals. RT0 instead uses flux, pressure and
private boundary-pressure coefficients, including the local normal-flux
constraint. The global `Equation` supplies the signed Dirichlet moments and
retains each constant pressure mode. Pure Neumann data add the physical pressure
integral through an explicit constraint. `define_darcy` returns equations;
`assemble`, the global solve and field interpretation are separate stages.


In [ ]:
from pymhm import assemble, SolverConfig
from examples.formulations.darcy import define_darcy, pressure_constraints, recover_darcy

mesh = TriangleMesh.unit_square(2)
natural = {int(face): 0.0 for face in mesh.boundary_faces}
with threadpool_limits(1):
    solution_definition = define_darcy(
        mesh,
        degree=2,
        local_refinement=2,
        neumann=natural,
        source=lambda x: 8 * np.pi**2 * np.prod(np.cos(2 * np.pi * x), axis=1),
        quadrature_order=10,
    )
    solution_system = assemble(solution_definition.problem)
    solution = recover_darcy(
        solution_definition,
        solution_system,
        solution_system.solve(
            constraints=pressure_constraints(solution_definition, solution_system)
        ),
    )
    indicator = estimate_darcy_jumps(solution, neumann=natural, calibration=3.0)
print({"jump_indicator": indicator.total, "macro_cells": len(mesh.cells)})


## Five-level numerical record

The archived sequences use n=4,8,16,32,64, four fine triangles per macro, and local/trace degree pairs (2,0) and (5,3). The paper's calibrations 3 and 50 are retained. The norm is a sum of a macro-diameter-weighted broken flux H(div) term, H-inverse pressure L2 error and broken gradient error. The constant-trace indicator lies below this finite-local error; the cubic-trace ratio tends to 1.684.


In [ ]:
rows = json.loads((root / "examples/results/darcy-jump/campaign.json").read_text())
for row in rows:
    print(row["trace_degree"], row["resolution"], row["paper_error_sum"], row["indicator"], row["effectivity"])
display(Image(filename=str(root / "docs/figures/darcy-jump/refinement.png")))
